# Delta Demo — Episode 12: Time Travel, RESTORE, and Selective Recovery
### "RESTORE Can Undo a Mistake — But What If New Data Arrived Since Then?"

---
**Prerequisites:** None

**Runtime:** Databricks Free Edition

**Run Mode:** Run All

**Safe to rerun:** Yes

**Creates its own demo table:** `sales_ep12`

**Deletes only its own demo data:** Yes

---

**This notebook is self-contained.** It creates and uses its own Delta table (`sales_ep12`) in a dedicated path — nothing outside this path is ever touched, and no other episode needs to be run first.

**Learning Outcome:** By the end of this episode, viewers should be able to explain why `RESTORE TABLE` can only revert an ENTIRE table to one past version, and how to selectively recover corrupted data using time travel reads combined with `CREATE OR REPLACE TABLE AS SELECT` — without losing legitimate data that arrived after the mistake.

**Core Question:** A bad UPDATE corrupted 20 rows. Then 7 more, completely correct, rows arrived. Can we fix the 20 without losing the 7?

### Today's Journey
✔ Build up sales data across three days

↓

✔ A sign-error UPDATE corrupts every existing row

↓

✔ Fresh, correct data arrives on top of the corruption

↓

✔ Realize why a plain RESTORE would destroy the new data too

↓

✔ Selectively recover: fix the old rows, keep the new rows, in one query

↓

✔ Confirm the mistake is still visible in history — just no longer active

# =====================================================
# STEP 0 — Setup (Self-Contained Reset)
# =====================================================

In [0]:
%sh
rm -rf /Volumes/workspace/delta_demo/demo_files/sales_ep12

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.delta_demo;
CREATE VOLUME IF NOT EXISTS workspace.delta_demo.demo_files;

# =====================================================
# STEP 1 — Day 1: Create Baseline (10 Records, Jan 1)
# =====================================================

In [0]:
%python
from pyspark.sql import functions as F
import glob, json

table_path = "/Volumes/workspace/delta_demo/demo_files/sales_ep12"

products = ['Widget A', 'Widget B', 'Widget C', 'Widget D']
regions = ['East', 'West', 'North', 'South']

day1_rows = [
    (i, products[(i-1) % 4], regions[(i-1) % 4], '2026-01-01', float(100 + i * 10))
    for i in range(1, 11)
]

day1 = spark.createDataFrame(
    day1_rows,
    "sale_id INT, product_name STRING, region STRING, sale_date STRING, amount DOUBLE"
).withColumn("sale_date", F.col("sale_date").cast("DATE")) \
 .withColumn("amount", F.col("amount").cast("DECIMAL(10,2)"))

day1.write.format("delta").mode("overwrite").save("/Volumes/workspace/delta_demo/demo_files/sales_ep12")

### Verify Day 1

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,110.00
2,Widget B,West,2026-01-01,120.00
3,Widget C,North,2026-01-01,130.00
4,Widget D,South,2026-01-01,140.00
5,Widget A,East,2026-01-01,150.00
6,Widget B,West,2026-01-01,160.00
7,Widget C,North,2026-01-01,170.00
8,Widget D,South,2026-01-01,180.00
9,Widget A,East,2026-01-01,190.00
10,Widget B,West,2026-01-01,200.00


# =====================================================
# STEP 2 — Day 2: Insert 5 Records (Jan 2)
# =====================================================

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VALUES
(11, 'Widget A', 'East',  DATE'2026-01-02', 210.00),
(12, 'Widget B', 'West',  DATE'2026-01-02', 220.00),
(13, 'Widget C', 'North', DATE'2026-01-02', 230.00),
(14, 'Widget D', 'South', DATE'2026-01-02', 240.00),
(15, 'Widget A', 'East',  DATE'2026-01-02', 250.00);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT COUNT(*) AS row_count FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

row_count
15


# =====================================================
# STEP 3 — Day 3: Insert 5 More Records (Jan 3) — 20 Rows Total
# =====================================================
**This is the clean state we'll need to recover back to later — capture its version number now, programmatically.**

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VALUES
(16, 'Widget B', 'West',  DATE'2026-01-03', 260.00),
(17, 'Widget C', 'North', DATE'2026-01-03', 270.00),
(18, 'Widget D', 'South', DATE'2026-01-03', 280.00),
(19, 'Widget A', 'East',  DATE'2026-01-03', 290.00),
(20, 'Widget B', 'West',  DATE'2026-01-03', 300.00);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`")
clean_version = history_df.orderBy(history_df.version.desc()).first()['version']
print(f"Clean 20-row version (before any mistake): {clean_version}")

Clean 20-row version (before any mistake): 2


In [0]:
%sql
SELECT COUNT(*) AS row_count FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

row_count
20


# =====================================================
# STEP 4 — The Mistake: Sign-Error UPDATE on ALL 20 Rows
# =====================================================
A finance adjustment was meant to be `+10%`. A sign error made it `-10%`, applied with no WHERE clause — every existing row is now wrong.

In [0]:
%sql
UPDATE delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` SET amount = amount * -1.10;

num_affected_rows
20


### The Business Symptom

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,-121.00
2,Widget B,West,2026-01-01,-132.00
3,Widget C,North,2026-01-01,-143.00
4,Widget D,South,2026-01-01,-154.00
5,Widget A,East,2026-01-01,-165.00
6,Widget B,West,2026-01-01,-176.00
7,Widget C,North,2026-01-01,-187.00
8,Widget D,South,2026-01-01,-198.00
9,Widget A,East,2026-01-01,-209.00
10,Widget B,West,2026-01-01,-220.00


In [0]:
%python
negative_count = spark.read.format("delta").load("/Volumes/workspace/delta_demo/demo_files/sales_ep12") \
    .filter(F.col("amount") < 0).count()
print(f"Rows with negative revenue right now: {negative_count}")

Rows with negative revenue right now: 20


# =====================================================
# STEP 5 — Day 4: 7 Fresh, Correct Records Arrive (Jan 4)
# =====================================================
Nobody has noticed the mistake yet. New data keeps flowing in normally.

In [0]:
%sql
INSERT INTO delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VALUES
(21, 'Widget C', 'North', DATE'2026-01-04', 310.00),
(22, 'Widget D', 'South', DATE'2026-01-04', 320.00),
(23, 'Widget A', 'East',  DATE'2026-01-04', 330.00),
(24, 'Widget B', 'West',  DATE'2026-01-04', 340.00),
(25, 'Widget C', 'North', DATE'2026-01-04', 350.00),
(26, 'Widget D', 'South', DATE'2026-01-04', 360.00),
(27, 'Widget A', 'East',  DATE'2026-01-04', 370.00);

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
SELECT COUNT(*) AS row_count FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

row_count
27


**27 rows now: 20 corrupted (negative amounts) + 7 correct (Jan 4).** This is the moment the problem gets discovered.

# =====================================================
# STEP 6 — Why a Plain RESTORE Won't Work Here
# =====================================================
If we ran `RESTORE TABLE ... TO VERSION AS OF <clean_version>` right now, it would bring back the correct 20 rows — but it would also **completely erase the 7 Jan 4 rows**, since `RESTORE` reverts the ENTIRE table to a single snapshot. There is no "restore only some rows" option.

In [0]:
%python
# Prove this concretely: how many rows existed at the clean version?
clean_snapshot = spark.read.format("delta") \
    .option("versionAsOf", clean_version).load("/Volumes/workspace/delta_demo/demo_files/sales_ep12")
print(f"Rows at the clean version ({clean_version}): {clean_snapshot.count()}")
print(f"Rows in the CURRENT table right now: {spark.read.format('delta').load("/Volumes/workspace/delta_demo/demo_files/sales_ep12").count()}")
print(f"\nA full RESTORE to version {clean_version} would drop the table")
print(f"from 27 rows back down to {clean_snapshot.count()} — losing all 7 Jan 4 rows.")

Rows at the clean version (2): 20
Rows in the CURRENT table right now: 27

A full RESTORE to version 2 would drop the table
from 27 rows back down to 20 — losing all 7 Jan 4 rows.


# =====================================================
# STEP 7 — Selective Recovery: Fix the Old Rows, Keep the New Rows
# =====================================================
One query, two sources: read the OLD clean snapshot (time travel) for the 20 rows, apply the correct `+10%` to them, and combine with the CURRENT table's Jan 4 rows — untouched. This is pure SQL, same `CREATE OR REPLACE TABLE AS SELECT` mechanism from Episode 11, just fed by two sources instead of one.

In [0]:
%python
# Build the query using the real captured version number — never hardcoded.
reconstruction_sql = f"""
CREATE OR REPLACE TABLE delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` AS
SELECT sale_id, product_name, region, sale_date, amount * 1.10 AS amount
FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF {clean_version}

UNION ALL

SELECT sale_id, product_name, region, sale_date, amount
FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`
WHERE sale_date = '2026-01-04'
"""
print(reconstruction_sql)


CREATE OR REPLACE TABLE delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` AS
SELECT sale_id, product_name, region, sale_date, amount * 1.10 AS amount
FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 2

UNION ALL

SELECT sale_id, product_name, region, sale_date, amount
FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`
WHERE sale_date = '2026-01-04'



In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-02T01:12:45.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),634ddd95-d8f6-4c92-bab6-c026c0c858c6,0802-001409-h47590in-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 7, numOutputBytes -> 1841)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T01:12:00.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,Map(predicate -> []),null,List(1258115999882544),a174ada9-b63f-4094-bc59-233b56dbb63d,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5462, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 781, numDeletionVectorsUpdated -> 0, scanTimeMs -> 7, numAddedFiles -> 1, numUpdatedRows -> 20, numAddedBytes -> 2015, rewriteTimeMs -> 774)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T01:11:03.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),08ce4506-0987-4537-9b3a-eabb0133657b,0802-001409-h47590in-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T01:10:43.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),9951185f-ab2c-4719-a9bf-08736de383c9,0802-001409-h47590in-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T01:10:16.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),967e4805-7fdd-4b05-a4e6-90d801cecbe3,0802-001409-h47590in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 1910)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


### Run the Reconstruction

In [0]:
%python
spark.sql(reconstruction_sql)

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

# =====================================================
# STEP 8 — Verify Everything
# =====================================================

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,121.0000
2,Widget B,West,2026-01-01,132.0000
3,Widget C,North,2026-01-01,143.0000
4,Widget D,South,2026-01-01,154.0000
5,Widget A,East,2026-01-01,165.0000
6,Widget B,West,2026-01-01,176.0000
7,Widget C,North,2026-01-01,187.0000
8,Widget D,South,2026-01-01,198.0000
9,Widget A,East,2026-01-01,209.0000
10,Widget B,West,2026-01-01,220.0000


### VERIFY — 27 Rows, No Negative Amounts, Jan 4 Untouched

In [0]:
%python
final_df = spark.read.format("delta").load("/Volumes/workspace/delta_demo/demo_files/sales_ep12")
final_count = final_df.count()
negative_count_final = final_df.filter(F.col("amount") < 0).count()
jan4_count = final_df.filter(F.col("sale_date") == "2026-01-04").count()

print(f"Total rows: {final_count}")
print(f"Rows with negative amount: {negative_count_final}")
print(f"Jan 4 rows present: {jan4_count}")

if final_count == 27 and negative_count_final == 0 and jan4_count == 7:
    print("\n✅ VERIFIED: all 27 rows present, zero negative amounts,")
    print("   all 7 Jan 4 rows intact and untouched.")
else:
    print("\n❌ NOT VERIFIED — investigate above.")

Total rows: 27
Rows with negative amount: 0
Jan 4 rows present: 7

✅ VERIFIED: all 27 rows present, zero negative amounts,
   all 7 Jan 4 rows intact and untouched.


### VERIFY — The Mistake Is Still in History (Just Not Active)

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-02T01:19:01.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1258115999882544),5d0052ba-c73b-4a85-bab1-49415174d14e,0802-001409-h47590in-v2n,4,WriteSerializable,false,"Map(numFiles -> 2, numRemovedFiles -> 2, numRemovedBytes -> 3856, numDeletionVectorsRemoved -> 0, numOutputRows -> 27, numOutputBytes -> 3909)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T01:12:45.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),634ddd95-d8f6-4c92-bab6-c026c0c858c6,0802-001409-h47590in-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 7, numOutputBytes -> 1841)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T01:12:00.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,Map(predicate -> []),null,List(1258115999882544),a174ada9-b63f-4094-bc59-233b56dbb63d,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5462, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 781, numDeletionVectorsUpdated -> 0, scanTimeMs -> 7, numAddedFiles -> 1, numUpdatedRows -> 20, numAddedBytes -> 2015, rewriteTimeMs -> 774)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T01:11:03.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),08ce4506-0987-4537-9b3a-eabb0133657b,0802-001409-h47590in-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T01:10:43.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),9951185f-ab2c-4719-a9bf-08736de383c9,0802-001409-h47590in-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T01:10:16.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),967e4805-7fdd-4b05-a4e6-90d801cecbe3,0802-001409-h47590in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 1910)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


In [0]:
%python
history_df = spark.sql(f"DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`")
min_version = history_df.agg(F.min("version")).collect()[0][0]
total_versions = history_df.count()
update_commits = history_df.filter(history_df.operation == "UPDATE").count()

print(f"Total versions in history: {total_versions}")
print(f"UPDATE commits found (the mistake): {update_commits}")

if min_version == 0 and update_commits >= 1:
    print("\n✅ VERIFIED: full history intact from version 0 — the mistake's")
    print("   commit is still there permanently, even though the table's")
    print("   CURRENT state no longer reflects it.")
else:
    print("\n❌ Unexpected — investigate.")

Total versions in history: 6
UPDATE commits found (the mistake): 1

✅ VERIFIED: full history intact from version 0 — the mistake's
   commit is still there permanently, even though the table's
   CURRENT state no longer reflects it.


# =====================================================
# STEP 9 — Enterprise Reality
# =====================================================
##> "`RESTORE` is the right tool when an entire table needs to go back to one point in time. But real pipelines rarely stop moving just because a mistake happened — new, legitimate data keeps arriving. Selective recovery, combining a time-travel read of the old state with the current table's untouched rows, is how enterprises fix a mistake without losing everything that happened after it."

**We fixed 20 corrupted rows and kept 7 correct ones, in a single query, without ever using RESTORE.**

The mistake is still sitting in `DESCRIBE HISTORY`, permanently, as version 3 — proof that Delta never actually deletes anything just because you fixed it forward.

But how long does that old, corrupted data actually stay recoverable? That's exactly what we'll explore in the next episode: VACUUM, and what happens once the retention window runs out.

### Version 0 — Day 1 Baseline (10 Records)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 0 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,110.00
2,Widget B,West,2026-01-01,120.00
3,Widget C,North,2026-01-01,130.00
4,Widget D,South,2026-01-01,140.00
5,Widget A,East,2026-01-01,150.00
6,Widget B,West,2026-01-01,160.00
7,Widget C,North,2026-01-01,170.00
8,Widget D,South,2026-01-01,180.00
9,Widget A,East,2026-01-01,190.00
10,Widget B,West,2026-01-01,200.00


### Version 1 — Day 2 (+5 Records, 15 Total)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 1 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,110.00
2,Widget B,West,2026-01-01,120.00
3,Widget C,North,2026-01-01,130.00
4,Widget D,South,2026-01-01,140.00
5,Widget A,East,2026-01-01,150.00
6,Widget B,West,2026-01-01,160.00
7,Widget C,North,2026-01-01,170.00
8,Widget D,South,2026-01-01,180.00
9,Widget A,East,2026-01-01,190.00
10,Widget B,West,2026-01-01,200.00


### Version 2 — Day 3 (+5 More Records, 20 Total — The Clean Snapshot)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 2 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,110.00
2,Widget B,West,2026-01-01,120.00
3,Widget C,North,2026-01-01,130.00
4,Widget D,South,2026-01-01,140.00
5,Widget A,East,2026-01-01,150.00
6,Widget B,West,2026-01-01,160.00
7,Widget C,North,2026-01-01,170.00
8,Widget D,South,2026-01-01,180.00
9,Widget A,East,2026-01-01,190.00
10,Widget B,West,2026-01-01,200.00


### Version 3 — The Sign-Error UPDATE (All 20 Rows Corrupted)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 3 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,-121.00
2,Widget B,West,2026-01-01,-132.00
3,Widget C,North,2026-01-01,-143.00
4,Widget D,South,2026-01-01,-154.00
5,Widget A,East,2026-01-01,-165.00
6,Widget B,West,2026-01-01,-176.00
7,Widget C,North,2026-01-01,-187.00
8,Widget D,South,2026-01-01,-198.00
9,Widget A,East,2026-01-01,-209.00
10,Widget B,West,2026-01-01,-220.00


### Version 4 — Day 4 (+7 Fresh Records, 27 Total — Mixed Good and Bad)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 4 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,-121.00
2,Widget B,West,2026-01-01,-132.00
3,Widget C,North,2026-01-01,-143.00
4,Widget D,South,2026-01-01,-154.00
5,Widget A,East,2026-01-01,-165.00
6,Widget B,West,2026-01-01,-176.00
7,Widget C,North,2026-01-01,-187.00
8,Widget D,South,2026-01-01,-198.00
9,Widget A,East,2026-01-01,-209.00
10,Widget B,West,2026-01-01,-220.00


### Version 5 — After Reconstruction (27 Rows, All Correct)

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 5 ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,121.0000
2,Widget B,West,2026-01-01,132.0000
3,Widget C,North,2026-01-01,143.0000
4,Widget D,South,2026-01-01,154.0000
5,Widget A,East,2026-01-01,165.0000
6,Widget B,West,2026-01-01,176.0000
7,Widget C,North,2026-01-01,187.0000
8,Widget D,South,2026-01-01,198.0000
9,Widget A,East,2026-01-01,209.0000
10,Widget B,West,2026-01-01,220.0000


### Current State — Latest Version, No Hardcoded Number

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` ORDER BY sale_id;

sale_id,product_name,region,sale_date,amount
1,Widget A,East,2026-01-01,121.0000
2,Widget B,West,2026-01-01,132.0000
3,Widget C,North,2026-01-01,143.0000
4,Widget D,South,2026-01-01,154.0000
5,Widget A,East,2026-01-01,165.0000
6,Widget B,West,2026-01-01,176.0000
7,Widget C,North,2026-01-01,187.0000
8,Widget D,South,2026-01-01,198.0000
9,Widget A,East,2026-01-01,209.0000
10,Widget B,West,2026-01-01,220.0000


### Confirm Real Version Numbers Before Trusting Any of the Above

In [0]:
%sql
DESCRIBE HISTORY delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12`;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-08-02T01:19:01.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(1258115999882544),5d0052ba-c73b-4a85-bab1-49415174d14e,0802-001409-h47590in-v2n,4,WriteSerializable,false,"Map(numFiles -> 2, numRemovedFiles -> 2, numRemovedBytes -> 3856, numDeletionVectorsRemoved -> 0, numOutputRows -> 27, numOutputBytes -> 3909)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
4,2026-08-02T01:12:45.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),634ddd95-d8f6-4c92-bab6-c026c0c858c6,0802-001409-h47590in-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 7, numOutputBytes -> 1841)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-02T01:12:00.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,UPDATE,Map(predicate -> []),null,List(1258115999882544),a174ada9-b63f-4094-bc59-233b56dbb63d,0802-001409-h47590in-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 5462, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 781, numDeletionVectorsUpdated -> 0, scanTimeMs -> 7, numAddedFiles -> 1, numUpdatedRows -> 20, numAddedBytes -> 2015, rewriteTimeMs -> 774)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-02T01:11:03.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),08ce4506-0987-4537-9b3a-eabb0133657b,0802-001409-h47590in-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-02T01:10:43.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),9951185f-ab2c-4719-a9bf-08736de383c9,0802-001409-h47590in-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1776)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-02T01:10:16.000Z,72738353004153,srikanth.enterprise.ai@gmail.com,WRITE,"Map(mode -> Overwrite, statsOnLoad -> false, partitionBy -> [])",null,List(1258115999882544),967e4805-7fdd-4b05-a4e6-90d801cecbe3,0802-001409-h47590in-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 1910)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13


### Intentionally Query a Version That Doesn't Exist Yet

In [0]:
%sql
SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 6 ORDER BY sale_id;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8628893205251466>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * FROM delta.`/Volumes/workspace/delta_demo/demo_files/sales_ep12` VERSION AS OF 6 ORDER BY sale_id;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    2